<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%203/1g_Pandas_Joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Combining Data Frames



## Setup

In [ ]:
!pip install -U faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from faker import Faker


In [ ]:
# Initialize Faker
Faker.seed(42)
fake = Faker('en_US')
type(fake)


faker.proxy.Faker

In [ ]:
# Number of entries you want
num_entries = 6
num_entries


6

In [ ]:
column_generators = {
            'ID' : lambda: None,
    'First Name' : fake.first_name,
   'Middle Name' : lambda: fake.first_name() if fake.boolean(chance_of_getting_true=75) else None,
     'Last Name' : fake.last_name,
       'Address' : lambda: fake.address().replace('\n', ', '),
  'Phone_Number' : fake.phone_number,
         'Title' : fake.job,
       'Company' : fake.company,
}
column_generators


{'ID': <function __main__.<lambda>()>,
 'First Name': <bound method Provider.first_name of <faker.providers.person.en_US.Provider object at 0x7af7255cfc50>>,
 'Middle Name': <function __main__.<lambda>()>,
 'Last Name': <bound method Provider.last_name of <faker.providers.person.en_US.Provider object at 0x7af7255cfc50>>,
 'Address': <function __main__.<lambda>()>,
 'Phone_Number': <bound method Provider.phone_number of <faker.providers.phone_number.en_US.Provider object at 0x7af7261ae990>>,
 'Title': <bound method Provider.job of <faker.providers.job.en_US.Provider object at 0x7af7250ccce0>>,
 'Company': <bound method Provider.company of <faker.providers.company.en_US.Provider object at 0x7af7250e0ce0>>}

In [ ]:
df = pd.DataFrame( {
  col_name: [ generator() for _ in range(num_entries) ]
    for col_name, generator in column_generators.items()
})
df["ID"] = (df.index + 1000).astype('Int64')
df


,ID,First Name,Middle Name,Last Name,Address,Phone_Number,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,Chief Strategy Officer,"Brown, Valdez and Lucas"
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,Plant breeder/geneticist,Powell LLC
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Manufacturing systems engineer,Wright and Sons
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Pharmacologist,"West, Henderson and Ramirez"
4,1004,Jill,James,Booth,"22691 James Mountain, Tashatown, TX 94967",603-910-5183,Paediatric nurse,Baxter Inc
5,1005,Erica,None,Blake,"514 Moore Alley Suite 828, Port Colleenhaven, ...",582.399.7376,"Psychologist, sport and exercise",Morton-Chase


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ID            6 non-null      Int64 
 1   First Name    6 non-null      object
 2   Middle Name   5 non-null      object
 3   Last Name     6 non-null      object
 4   Address       6 non-null      object
 5   Phone_Number  6 non-null      object
 6   Title         6 non-null      object
 7   Company       6 non-null      object
dtypes: Int64(1), object(7)
memory usage: 522.0+ bytes


In [ ]:
personal_df = df[
 ['ID',
 'First Name',
 'Middle Name',
 'Last Name',
 'Address',
 'Phone_Number',
 ]
][:4]
personal_df


,ID,First Name,Middle Name,Last Name,Address,Phone_Number
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983


In [ ]:
work_df = df[
 ['ID',
 'First Name',
 'Middle Name',
 'Last Name',
 'Title',
 'Company']
][2:].assign(_=lambda x: x.index + 100).set_index('_')
work_df


,ID,First Name,Middle Name,Last Name,Title,Company
_,,,,,,
102,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
103,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
104,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
105,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


## .concat(), .join(), .merge()

The three pandas functions —`.concat()`, `.join()`, and `.merge()`— combine data frames, but differ primarily in their **purpose** (stacking vs. lookup) and the **axes** (row labels/index vs. columns) they use for alignment.

When specifying the data frame, the first data frame is termed the "left" and the second data frame is termed the "right".

### Summary of Differences

| Function | Primary Role | Default Alignment Key | Flexibility/Control |
| :--- | :--- | :--- | :--- |
| **`pd.concat()`** | Stacking/Appending | Opposite Axis (Index or Columns) | Good for simple structural combination. |
| **`.join()`** | Index Lookup | **Index** | Simple and fast for index-to-index lookup. |
| **`pd.merge()`** | Database-Style Join | **Specific Columns** (Keys) | Highest control over keys and join type. |

## Concatenate

Using the `pd.concat()` method.




### Defaults


#### axis
By default, data frames are stacked one on top of each other, i.e. `axis = 0`.  Where column names match, those columns are used.  Where they do not match, the new columns are added to the right.



In [ ]:
pd.concat([personal_df, work_df])

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,NaN,NaN
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,NaN,NaN
102,1002,Joshua,Jeffery,Cole,NaN,NaN,Manufacturing systems engineer,Wright and Sons
103,1003,Jeffrey,Carolyn,Ramirez,NaN,NaN,Pharmacologist,"West, Henderson and Ramirez"
104,1004,Jill,James,Booth,NaN,NaN,Paediatric nurse,Baxter Inc
105,1005,Erica,None,Blake,NaN,NaN,"Psychologist, sport and exercise",Morton-Chase


### Side-by-side

To have data frames extend, use `axis = 1`.  The columns from the right data frame are added to the right.  By default rows are matched on index name.  If there is no match, new rows are added to the bottom.  Column values for rows that don't match are filled in with nulls.


In [ ]:
pd.concat([personal_df, work_df], axis=1)

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,ID,First Name,Middle Name,Last Name,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,<NA>,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,<NA>,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,<NA>,NaN,NaN,NaN,NaN,NaN
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,<NA>,NaN,NaN,NaN,NaN,NaN
102,<NA>,NaN,NaN,NaN,NaN,NaN,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
103,<NA>,NaN,NaN,NaN,NaN,NaN,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
104,<NA>,NaN,NaN,NaN,NaN,NaN,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
105,<NA>,NaN,NaN,NaN,NaN,NaN,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


### Ignoring the row index

This doesn't seem to work.

In [ ]:
pd.concat([personal_df, work_df], ignore_index=True , axis=1)

,0,1,2,3,4,5,6,7,8,9,10,11
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,<NA>,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,<NA>,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,<NA>,NaN,NaN,NaN,NaN,NaN
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,<NA>,NaN,NaN,NaN,NaN,NaN
102,<NA>,NaN,NaN,NaN,NaN,NaN,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
103,<NA>,NaN,NaN,NaN,NaN,NaN,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
104,<NA>,NaN,NaN,NaN,NaN,NaN,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
105,<NA>,NaN,NaN,NaN,NaN,NaN,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


Resetting the index so that the index of the right matches the index on the left extends the rows.  Notice that non-matching rows from either data frame have their columns filled with nulls.  Also notice that column names do not change, i.e. we now have duplicated column names.


In [ ]:
pd.concat(
  [personal_df,
    work_df
    .reset_index(drop=True)
    .assign(_=lambda x: x.index + 1)
    .set_index('_')
  ],
  axis=1
)

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,ID,First Name,Middle Name,Last Name,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,<NA>,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
4,<NA>,NaN,NaN,NaN,NaN,NaN,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


In [ ]:
foo = pd.concat([personal_df.sample(n=2).reset_index(drop=True), work_df.sample(n=2).reset_index(drop=True)], axis = 1)
foo

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,ID,First Name,Middle Name,Last Name,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
1,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons


In [ ]:
foo.columns

Index(['ID', 'First Name', 'Middle Name', 'Last Name', 'Address',
       'Phone_Number', 'ID', 'First Name', 'Middle Name', 'Last Name', 'Title',
       'Company'],
      dtype='object')

In [ ]:
foo.iloc[:,6]

,ID
0,1003
1,1002


## Join

The `.join()` method combines two tables based on row index.  When columns have the same name, columns from one of the data frame need different suffixes. Non-matching rows are excluded, even though columns are included.



In [ ]:
personal_df.join(work_df, rsuffix="_r")

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,ID_r,First Name_r,Middle Name_r,Last Name_r,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,<NA>,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,<NA>,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,<NA>,NaN,NaN,NaN,NaN,NaN
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,<NA>,NaN,NaN,NaN,NaN,NaN


In [ ]:
personal_df.join(work_df.reset_index(drop=True), rsuffix="_r")

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,ID_r,First Name_r,Middle Name_r,Last Name_r,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


Often columns are converted to indices.


In [ ]:
(
personal_df
  .set_index("ID")
  .join(
    work_df
    .set_index("ID"),
  rsuffix="_r")
).reset_index()

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,First Name_r,Middle Name_r,Last Name_r,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


## Merge



The `.merge()` method combines two tables using column names.

**Purpose:** The most powerful and flexible method, designed for **database-style joins** or lookups. It provides granular control over which columns are used for alignment and what type of join is performed.

| Feature | Description |
| :--- | :--- |
| **Default Axis** | **Column to Column** (or Column to Index) |
| **Alignment** | Aligns one or more **specific columns** (or the index) from one DataFrame with columns/index of the other. |
| **Use Case** | Complex joins based on common ID columns (e.g., joining an `Employees` table on `manager_id` to a `Departments` table on `department_id`). |



### Key Behaviors



* **Explicit Keys:** Requires you to specify the join key(s) using the `on`, `left_on`, `right_on`, `left_index`, and `right_index` parameters.


| Parameter Value (`how`) | Join Type | Logic and Result |
| :--- | :--- | :--- |
| **`cross`** | **Cartesian Product** | Generates a new DataFrame containing **all possible combinations** of rows. Every row from the left DataFrame is combined with every row from the right DataFrame. The keys specified by `on`, `left_on`, etc., are ignored. |
| **`inner`** (Default) | Inner Join | Keeps only records that have **matching keys** in **both** DataFrames. Rows that don't match on the key columns are discarded. |
| **`left`** | Left Outer Join | Keeps **all records** from the **left** DataFrame and the matching records from the right. Where there is no match in the right, columns are filled with missing values. |
| **`right`** | Right Outer Join | Keeps **all records** from the **right** DataFrame and the matching records from the left. Where there is no match in the left, columns are filled with missing values. |
| **`outer`** | Full Outer Join | Keeps **all records** when there is a match in one or both DataFrames. Missing matches in either DataFrame are filled with missing values. |






### how="cross" - Cross or Cartesian Join

This pairs ALL rows from the left data frame with ALL rows from the right data frame. This results in a table of row size p x q, where p is the number of rows in the left data frame and q is the number of rows in the right data frame.

#### Cross or Cartesian Join Example


In [ ]:
personal_df.merge(work_df, how="cross")

,ID_x,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,ID_y,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
2,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
3,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase
4,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
5,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
6,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
7,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase
8,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
9,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


In [ ]:
personal_df.merge(work_df, how='cross').shape

(16, 12)

In [ ]:
personal_df.shape, work_df.shape

((4, 6), (4, 6))

Very inefficient, but works.

In [ ]:
(
personal_df
  .merge(work_df,how="cross")
  .query("ID_x == ID_y", engine='python')
)

,ID_x,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,ID_y,First Name_y,Middle Name_y,Last Name_y,Title,Company
8,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
13,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


### how="inner" - Inner Join




#### Inner Join Example

Using all column names that match.

In [ ]:
personal_df.merge(work_df, how="inner")

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Manufacturing systems engineer,Wright and Sons
1,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Pharmacologist,"West, Henderson and Ramirez"


Using only one column.  Notice the renaming of columns with the same name.

In [ ]:
personal_df.merge(work_df, how="inner", on=["ID"])

,ID,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


In [ ]:
personal_df.merge(work_df[["ID","Title","Company"]], how="inner")

,ID,First Name,Middle Name,Last Name,Address,Phone_Number,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Manufacturing systems engineer,Wright and Sons
1,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Pharmacologist,"West, Henderson and Ramirez"


Using indices.

In [ ]:
display(personal_df)
display(work_df)

,ID,First Name,Middle Name,Last Name,Address,Phone_Number
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983


,ID,First Name,Middle Name,Last Name,Title,Company
_,,,,,,
102,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
103,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
104,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
105,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


In [ ]:
personal_df.merge(work_df,how="inner", left_index=True, right_index=True)

,ID_x,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,ID_y,First Name_y,Middle Name_y,Last Name_y,Title,Company


In [ ]:
personal_df.merge(work_df.reset_index(drop=True),how="inner", left_index=True, right_index=True)

,ID_x,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,ID_y,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


In [ ]:
personal_df.merge(work_df, how="inner", on=["ID","First Name"])

,ID,First Name,Middle Name_x,Last Name_x,Address,Phone_Number,Middle Name_y,Last Name_y,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


### how="left" - Left Outer Join

Similar to `.join()` method.

#### Left Outer Join Example

In [ ]:
personal_df.merge(work_df, how="left", on="ID")

,ID,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"


### how="right" - Right Outer Join


#### Right Outer Join Example

In [ ]:
personal_df.merge(work_df,how="right", on="ID")

,ID,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
1,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
2,1004,NaN,NaN,NaN,NaN,NaN,Jill,James,Booth,Paediatric nurse,Baxter Inc
3,1005,NaN,NaN,NaN,NaN,NaN,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase


In [ ]:
work_df.merge(personal_df,how="left", on="ID")

,ID,First Name_x,Middle Name_x,Last Name_x,Title,Company,First Name_y,Middle Name_y,Last Name_y,Address,Phone_Number
0,1002,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383
1,1003,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez",Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983
2,1004,Jill,James,Booth,Paediatric nurse,Baxter Inc,NaN,NaN,NaN,NaN,NaN
3,1005,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase,NaN,NaN,NaN,NaN,NaN


### how="outer" - Full Outer Join

Similar to `.concat()` method.


#### Full Outer Join Example

In [ ]:
personal_df.merge(work_df,how="outer", on="ID")

,ID,First Name_x,Middle Name_x,Last Name_x,Address,Phone_Number,First Name_y,Middle Name_y,Last Name_y,Title,Company
0,1000,Danielle,Christopher,Miles,"4235 Christopher Court Suite 594, Lake Stephen...",(252)388-0957,NaN,NaN,NaN,NaN,NaN
1,1001,Angel,Anthony,Barnes,"310 Kendra Common Apt. 164, Reidstad, GA 49021",(443)903-9117x182,NaN,NaN,NaN,NaN,NaN
2,1002,Joshua,Jeffery,Cole,"192 Frank Light Suite 835, East Lydiamouth, MO...",+1-624-889-6383,Joshua,Jeffery,Cole,Manufacturing systems engineer,Wright and Sons
3,1003,Jeffrey,Carolyn,Ramirez,"76724 John Points Suite 969, Coxberg, NY 65187",+1-557-287-1331x50983,Jeffrey,Carolyn,Ramirez,Pharmacologist,"West, Henderson and Ramirez"
4,1004,NaN,NaN,NaN,NaN,NaN,Jill,James,Booth,Paediatric nurse,Baxter Inc
5,1005,NaN,NaN,NaN,NaN,NaN,Erica,None,Blake,"Psychologist, sport and exercise",Morton-Chase
